# Erzeugte Wahlkreis-JSON-Dateien prüfen

Dieses Notebook liest nur die fertigen Dateien ein. Die fachliche Herleitung und der Vergleich mit den amtlichen Rändern stehen bereits Schritt für Schritt im Aufbereitungsnotebook. Hier geht es um Dateigröße, Struktur und offensichtliche Datenfehler vor der Übernahme in die App.

In [1]:
from pathlib import Path
import json
import math

import pandas as pd
from IPython.display import display


def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "scripts").is_dir() and (candidate / "package.json").is_file():
            return candidate
    raise RuntimeError("Das Notebook muss innerhalb des Repository-Ordners laufen.")


ROOT = find_repository_root()
GENERATED_DIRECTORY = ROOT / "scripts/data/generated"
FIRST_VOTES_JSON = GENERATED_DIRECTORY / "first_votes.json"
SECOND_VOTES_JSON = GENERATED_DIRECTORY / "second_votes.json"

## 1. Dateien und Größen

In [2]:
for path in (FIRST_VOTES_JSON, SECOND_VOTES_JSON):
    if not path.is_file():
        raise FileNotFoundError(f"Datei fehlt: {path}")

file_sizes = pd.DataFrame(
    {
        "file": [FIRST_VOTES_JSON.name, SECOND_VOTES_JSON.name],
        "MiB": [
            FIRST_VOTES_JSON.stat().st_size / 1024**2,
            SECOND_VOTES_JSON.stat().st_size / 1024**2,
        ],
    }
)
display(file_sizes)
print(f"Zusammen: {file_sizes['MiB'].sum():.2f} MiB")

,file,MiB
0,first_votes.json,11.800857
1,second_votes.json,25.131282


Zusammen: 36.93 MiB


## 2. JSON lesen und erste Zeilen ansehen

In [3]:
first_votes = pd.read_json(FIRST_VOTES_JSON)
second_votes = pd.read_json(SECOND_VOTES_JSON)

print(f"Erststimmen-Zeilen: {len(first_votes):,}")
print(f"Zweitstimmen-Zeilen: {len(second_votes):,}")
display(first_votes.head(20))
display(second_votes.head(20))

Erststimmen-Zeilen: 79,152
Zweitstimmen-Zeilen: 167,796


,districtId,state,gender,ageGroup,party,voteType,electionMethod,votes
0,1,Schleswig-Holstein,m,18-24,AfD,1,in-person,231.776843
1,1,Schleswig-Holstein,m,25-34,AfD,1,in-person,705.316696
2,1,Schleswig-Holstein,m,35-44,AfD,1,in-person,950.222207
3,1,Schleswig-Holstein,m,45-54,AfD,1,in-person,1741.783073
4,1,Schleswig-Holstein,m,55-64,AfD,1,in-person,774.870699
5,1,Schleswig-Holstein,m,65+,AfD,1,in-person,532.130022
6,1,Schleswig-Holstein,w,18-24,AfD,1,in-person,132.194506
7,1,Schleswig-Holstein,w,25-34,AfD,1,in-person,561.809190
8,1,Schleswig-Holstein,w,35-44,AfD,1,in-person,528.917689
9,1,Schleswig-Holstein,w,45-54,AfD,1,in-person,930.668872


,districtId,state,gender,ageGroup,party,voteType,electionMethod,votes
0,1,Schleswig-Holstein,m,18-24,AfD,2,in-person,253.364855
1,1,Schleswig-Holstein,m,25-34,AfD,2,in-person,743.726072
2,1,Schleswig-Holstein,m,35-44,AfD,2,in-person,955.961620
3,1,Schleswig-Holstein,m,45-54,AfD,2,in-person,1798.592232
4,1,Schleswig-Holstein,m,55-64,AfD,2,in-person,793.733206
5,1,Schleswig-Holstein,m,65+,AfD,2,in-person,586.838226
6,1,Schleswig-Holstein,w,18-24,AfD,2,in-person,173.811204
7,1,Schleswig-Holstein,w,25-34,AfD,2,in-person,532.530895
8,1,Schleswig-Holstein,w,35-44,AfD,2,in-person,593.565954
9,1,Schleswig-Holstein,w,45-54,AfD,2,in-person,963.937793


## 3. Erwartete Spalten und Wertebereiche

In [4]:
expected_columns = {
    "districtId",
    "state",
    "gender",
    "ageGroup",
    "party",
    "voteType",
    "electionMethod",
    "votes",
}

for label, frame, expected_vote_type in (
    ("first_votes", first_votes, 1),
    ("second_votes", second_votes, 2),
):
    missing = expected_columns - set(frame.columns)
    unexpected = set(frame.columns) - expected_columns
    print(label, {"missing": sorted(missing), "unexpected": sorted(unexpected)})
    assert not missing
    assert not unexpected
    assert set(frame["voteType"].astype(str)) == {str(expected_vote_type)}
    assert set(frame["gender"]) <= {"m", "w"}
    assert set(frame["ageGroup"]) <= {"18-24", "25-34", "35-44", "45-54", "55-64", "65+"}
    assert set(frame["electionMethod"]) <= {"postal", "in-person"}
    assert frame["districtId"].notna().all()
    assert (frame["districtId"] > 0).all()
    assert frame["votes"].notna().all()
    assert frame["votes"].map(math.isfinite).all()
    assert (frame["votes"] >= 0).all()

print("Grundlegende Wertebereiche sind gültig.")

first_votes {'missing': [], 'unexpected': []}
second_votes {'missing': [], 'unexpected': []}
Grundlegende Wertebereiche sind gültig.


## 4. Doppelte Detailzeilen suchen

In [5]:
key_columns = [
    "districtId",
    "state",
    "gender",
    "ageGroup",
    "party",
    "voteType",
    "electionMethod",
]

for label, frame in (("first_votes", first_votes), ("second_votes", second_votes)):
    duplicates = frame[frame.duplicated(key_columns, keep=False)]
    print(f"{label}: {len(duplicates):,} doppelte Zeilen")
    display(duplicates.head(20))
    assert duplicates.empty

first_votes: 0 doppelte Zeilen


,districtId,state,gender,ageGroup,party,voteType,electionMethod,votes


second_votes: 0 doppelte Zeilen


,districtId,state,gender,ageGroup,party,voteType,electionMethod,votes


## 5. Abdeckung ansehen

In [6]:
coverage = pd.DataFrame(
    {
        "first_votes": [
            first_votes["districtId"].nunique(),
            first_votes["state"].nunique(),
            first_votes["party"].nunique(),
        ],
        "second_votes": [
            second_votes["districtId"].nunique(),
            second_votes["state"].nunique(),
            second_votes["party"].nunique(),
        ],
    },
    index=["Wahlkreise", "Bundesländer", "Parteien/Sammelkategorien"],
)
display(coverage)

,first_votes,second_votes
Wahlkreise,299,299
Bundesländer,16,16
Parteien/Sammelkategorien,45,40


## 6. Beispiel eines Wahlkreises

In [7]:
sample_district = int(first_votes["districtId"].min())

display(
    first_votes[first_votes["districtId"] == sample_district]
    .groupby(["party", "electionMethod"], as_index=False)["votes"]
    .sum()
    .sort_values("votes", ascending=False)
    .head(30)
)

,party,electionMethod,votes
10,GRÜNE,in-person,31751.0
2,CDU,in-person,29743.0
14,SPD,in-person,29588.0
11,GRÜNE,postal,18480.0
3,CDU,postal,11978.0
15,SPD,postal,9339.0
16,SSW,in-person,9280.0
6,FDP,in-person,9034.0
0,AfD,in-person,7936.0
4,DIE LINKE,in-person,4701.0
